In [2]:
import pandas as pd

In [3]:
salary_df = pd.read_csv(
    "../data/ai_jobs_salaries_clean.csv"
)

ai_df = pd.read_excel(
    "../data/Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx"
)

In [5]:
print(salary_df.shape)
print(salary_df["isco_group_hint"].value_counts(dropna=False))

(71913, 17)
isco_group_hint
NaN                                                           40540
Software and applications developers                          12475
Business services and administration managers                  8335
Mathematicians, actuaries and statisticians                    7216
Physical and earth science professionals                       2852
Information and communications technology service managers      495
Name: count, dtype: int64


In [8]:
print(ai_df.shape)
print(ai_df[["ISCO_08" , "Title" , "mean_score_2025" , "potential25"]].head(15))

(3265, 21)
    ISCO_08                        Title  mean_score_2025       potential25
0      1111                  Legislators             0.31       Not Exposed
1      1111                  Legislators             0.31       Not Exposed
2      1111                  Legislators             0.31       Not Exposed
3      1111                  Legislators             0.31       Not Exposed
4      1111                  Legislators             0.31       Not Exposed
5      1111                  Legislators             0.31       Not Exposed
6      1111                  Legislators             0.31       Not Exposed
7      1111                  Legislators             0.31       Not Exposed
8      1112  Senior Government Officials             0.38  Minimal Exposure
9      1112  Senior Government Officials             0.38  Minimal Exposure
10     1112  Senior Government Officials             0.38  Minimal Exposure
11     1112  Senior Government Officials             0.38  Minimal Exposure
1

In [9]:
ai_df[["ISCO_08" , "Title" , "mean_score_2025" , "potential25"]].drop_duplicates().head(20)

,ISCO_08,Title,mean_score_2025,potential25
0,1111,Legislators,0.31,Not Exposed
8,1112,Senior Government Officials,0.38,Minimal Exposure
17,1113,Traditional Chiefs and Heads of Villages,0.23,Not Exposed
24,1114,Senior Officials of Special-interest Organizat...,0.37,Minimal Exposure
33,1120,Managing Directors and Chief Executives,0.38,Minimal Exposure
44,1211,Finance Managers,0.37,Minimal Exposure
52,1212,Human Resource Managers,0.36,Minimal Exposure
63,1213,Policy and Planning Managers,0.36,Not Exposed
72,1219,Business Services and Administration Managers ...,0.42,Exposed: Gradient 2
82,1221,Sales and Marketing Managers,0.41,Exposed: Gradient 2


In [20]:
isco_groups = salary_df["isco_group_hint"].dropna().unique()

for group in isco_groups :
    print("\n" , "=" * 70)
    print(group)
    
    matches = ai_df[
        ai_df["label1d"].astype(str).str.contains(
            group.split()[0],
            case=False,
            na=False
        )
        |
        ai_df["Title"].astype(str).str.contains(
            group.split()[0],
            case = False ,
            na = False
        )
    ]
    
    result = (
        matches[
            [
            "ISCO_08", 
            "Title", 
            "mean_score_2025", 
            "potential25"
            ]
        ]
        .drop_duplicates()
        .sort_values("ISCO_08")
    )
    
    print(result.to_string(index=False))


Information and communications technology service managers
 ISCO_08                                                              Title  mean_score_2025         potential25
    1330         Information and Communications Technology Service Managers             0.44 Exposed: Gradient 2
    2356                                    Information Technology Trainers             0.47 Exposed: Gradient 2
    2434      Information and Communications Technology Sales Professionals             0.51 Exposed: Gradient 3
    2622                   Librarians and Related Information Professionals             0.55 Exposed: Gradient 3
    3252                 Medical Records and Health Information Technicians             0.52 Exposed: Gradient 3
    3511   Information and Communications Technology Operations Technicians             0.43 Exposed: Gradient 2
    3512 Information and Communications Technology User Support Technicians             0.47 Exposed: Gradient 2
    4222                            

In [21]:
isco_mapping = pd.DataFrame({
    "isco_group_hint": [
        "Information and communications technology service managers",
        "Mathematicians, actuaries and statisticians",
        "Software and applications developers",
        "Business services and administration managers",
        "Physical and earth science professionals"
    ],
    
    "ISCO_08": [
        1330,
        2120,
        2519,
        1219,
        None
    ]
})

isco_mapping

,isco_group_hint,ISCO_08
0,Information and communications technology serv...,1330.0
1,"Mathematicians, actuaries and statisticians",2120.0
2,Software and applications developers,2519.0
3,Business services and administration managers,1219.0
4,Physical and earth science professionals,NaN


In [22]:
ai_occupation = (
    ai_df[
        [
            "ISCO_08",
            "Title",
            "mean_score_2025",
            "SD_2025",
            "potential25"
        ]
    ]
    .drop_duplicates("ISCO_08")
)

isco_mapping = isco_mapping.merge(
    ai_occupation,
    on = "ISCO_08",
    how = "left"
    
)

isco_mapping

,isco_group_hint,ISCO_08,Title,mean_score_2025,SD_2025,potential25
0,Information and communications technology serv...,1330.0,Information and Communications Technology Serv...,0.44,0.07,Exposed: Gradient 2
1,"Mathematicians, actuaries and statisticians",2120.0,"Mathematicians, Actuaries and Statisticians",0.56,0.09,Exposed: Gradient 3
2,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
3,Business services and administration managers,1219.0,Business Services and Administration Managers ...,0.42,0.10,Exposed: Gradient 2
4,Physical and earth science professionals,NaN,NaN,NaN,NaN,NaN


In [23]:
merged_df = salary_df.merge(
    isco_mapping ,
    on = "isco_group_hint",
    how = "left"
)

merged_df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,employment_type_label,work_mode,salary_outlier_flag,role_family,isco_group_hint,ISCO_08,Title,mean_score_2025,SD_2025,potential25
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,...,Full-time,On-site,True,Analytics Manager,Information and communications technology serv...,1330.0,Information and Communications Technology Serv...,0.44,0.07,Exposed: Gradient 2
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,...,Full-time,On-site,False,Analytics Manager,Information and communications technology serv...,1330.0,Information and Communications Technology Serv...,0.44,0.07,Exposed: Gradient 2
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,...,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians",2120.0,"Mathematicians, Actuaries and Statisticians",0.56,0.09,Exposed: Gradient 3
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,...,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians",2120.0,"Mathematicians, Actuaries and Statisticians",0.56,0.09,Exposed: Gradient 3
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,...,Full-time,Remote,False,Other / Unclassified,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
merged_df.shape

(71913, 22)

In [25]:
merged_df["mean_score_2025"].notna().sum()

28521

In [27]:
coverage = (
    merged_df["mean_score_2025"].notna().mean() * 100
)

print(f"AI Exposure coverage: {coverage:.2f}%")

AI Exposure coverage: 39.66%


In [28]:
merged_df["potential25"].value_counts(dropna=False)

potential25
NaN                    43392
Exposed: Gradient 3    19691
Exposed: Gradient 2     8830
Name: count, dtype: int64

In [32]:
merged_df[
        (merged_df["job_title"]=="AI Architect")
    ]

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,employment_type_label,work_mode,salary_outlier_flag,role_family,isco_group_hint,ISCO_08,Title,mean_score_2025,SD_2025,potential25
2322,2025,SE,FT,AI Architect,250000,USD,250000,US,100,US,...,Full-time,Remote,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
2323,2025,SE,FT,AI Architect,124000,USD,124000,US,100,US,...,Full-time,Remote,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
2403,2025,SE,FT,AI Architect,210000,USD,210000,US,0,US,...,Full-time,On-site,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
2404,2025,SE,FT,AI Architect,140000,USD,140000,US,0,US,...,Full-time,On-site,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
2795,2025,SE,FT,AI Architect,272300,USD,272300,US,0,US,...,Full-time,On-site,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67566,2023,SE,FT,AI Architect,146900,USD,146900,US,100,US,...,Full-time,Remote,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
68697,2023,SE,FT,AI Architect,330000,USD,330000,US,0,US,...,Full-time,On-site,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
68698,2023,SE,FT,AI Architect,204000,USD,204000,US,0,US,...,Full-time,On-site,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3
69240,2023,EX,FT,AI Architect,200000,EUR,215936,BE,100,US,...,Full-time,Remote,False,AI Architect,Software and applications developers,2519.0,Software and Applications Developers and Analy...,0.55,0.16,Exposed: Gradient 3


In [33]:
merged_df.to_csv(
    "../data/ai_jobs_salaries_with_exposure.csv",
    index=False
)